# Project 1 — Sales and Profitability Analysis

Analyze sales performance using Python and pandas. The question is not simply “how much did we sell?” but which categories, regions, and channels drive revenue and profit. Data are synthetic for practice.

## Load and understand the data

Each row is a transaction line, not necessarily a full order. Read the data dictionary and inspect shape, data types, missingness, and duplicates before calculating KPIs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path  # Find the data folder robustly.
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")  # Support running from the project root or notebooks folder.
df = pd.read_csv(DATA_DIR / "sales_lines.csv", parse_dates=["order_date"])  # Load sales lines and parse dates.
print("Shape:", df.shape)  # Display number of rows and columns.
print(df.head())  # Preview sample records.
print("\nMissing values:\n", df.isna().sum())  # Check missingness in every field.
print("\nDuplicate rows:", df.duplicated().sum())  # Count exact duplicate rows.
print("\nUnique orders:", df["order_id"].nunique(), "| transaction lines:", len(df))  # Clarify that an order can have multiple lines.

## Create core business KPIs

Build metrics that connect sales volume and financial outcome. Revenue alone does not describe profitability; calculate gross profit and margin and keep distinct orders separate from line counts.

In [ ]:
df["gross_margin"] = df["gross_profit"] / df["revenue"].replace(0, np.nan)  # Calculate line-level margin safely.
total_revenue = df["revenue"].sum()  # Sum revenue over all transaction lines.
total_cost = df["cost"].sum()  # Sum cost over all transaction lines.
total_profit = df["gross_profit"].sum()  # Sum gross profit over all transaction lines.
unique_orders = df["order_id"].nunique()  # Count distinct orders, not transaction lines.
kpis = pd.Series({  # Assemble a compact KPI summary.
    "Revenue": total_revenue,  # Monetary sales total.
    "Cost": total_cost,  # Cost of sold items in this simplified dataset.
    "Gross profit": total_profit,  # Revenue less cost.
    "Gross margin": total_profit / total_revenue,  # Profit divided by revenue.
    "Unique orders": unique_orders,  # Distinct order count.
    "Average order value (line revenue proxy)": total_revenue / unique_orders,  # Revenue per unique order across all lines.
})  # Finish KPI calculations.
print(kpis.round(3))  # Display summary metrics.

## Find category and channel drivers

Compare revenue, profit, and margin by category and channel. A large revenue category can still be a weak profit contributor, so rank using more than one measure.

In [ ]:
category_perf = df.groupby("category").agg(  # Aggregate performance by product category.
    revenue=("revenue", "sum"),  # Sum category revenue.
    gross_profit=("gross_profit", "sum"),  # Sum category gross profit.
    quantity=("quantity", "sum"),  # Sum units sold.
).sort_values("revenue", ascending=False)  # Sort categories by revenue.
category_perf["gross_margin"] = category_perf["gross_profit"] / category_perf["revenue"]  # Calculate margin for each category.
channel_perf = df.groupby("channel").agg(  # Aggregate performance by channel.
    revenue=("revenue", "sum"),  # Sum channel revenue.
    gross_profit=("gross_profit", "sum"),  # Sum channel profit.
    line_count=("order_id", "size"),  # Count sales lines in each channel.
)  # Finish channel aggregation.
channel_perf["gross_margin"] = channel_perf["gross_profit"] / channel_perf["revenue"]  # Calculate channel margin.
print("By category:\n", category_perf.round(3))  # Show category performance.
print("\nBy channel:\n", channel_perf.round(3))  # Show channel performance.

## Trend analysis and visualization

Monthly trends help identify seasonality and changes. Use complete date periods and annotate comparisons with a baseline; do not treat every fluctuation as a cause.

In [ ]:
monthly = df.set_index("order_date").resample("MS").agg(  # Aggregate records by calendar month start.
    revenue=("revenue", "sum"),  # Sum monthly revenue.
    gross_profit=("gross_profit", "sum"),  # Sum monthly gross profit.
)  # Finish monthly aggregation.
monthly["gross_margin"] = monthly["gross_profit"] / monthly["revenue"]  # Calculate monthly gross margin.
ax = monthly[["revenue", "gross_profit"]].plot(figsize=(9, 4), title="Monthly Revenue and Gross Profit", marker="o")  # Plot monetary trends.
ax.set_xlabel("Month")  # Label the time axis.
ax.set_ylabel("Currency units")  # Label the monetary axis.
plt.tight_layout()  # Adjust plot spacing.
plt.show()  # Display the monthly chart.
print(monthly.round(2))  # Display data behind the visualization.

## Convert findings into a decision

A portfolio-quality analysis ends with an actionable recommendation, a metric for success, an owner, and a caveat. Avoid claiming the data proves a cause when it only reveals an association.

In [ ]:
top_profit_category = category_perf["gross_profit"].idxmax()  # Find the category with the highest gross profit.
lowest_margin_channel = channel_perf["gross_margin"].idxmin()  # Find the channel with the lowest gross margin.
print("Category generating the most gross profit:", top_profit_category)  # State the strongest profit contributor.
print("Channel with the lowest gross margin:", lowest_margin_channel)  # Flag the channel to investigate.
print("Next step: inspect discounts, product mix, returns, and fulfillment cost before recommending a change.")  # Suggest validation analyses.
print("Portfolio deliverable: dashboard + 3 evidence-backed findings + 2 recommendations + data limitations.")  # Remind what to publish.